# Perkalian Matriks Paralel (Tugas KPT)
Jalankan sel dari atas ke bawah (Runtime > Run all).
Catatan: Colab gratis biasanya hanya punya 2 core, jadi speedup maksimal sekitar 2x. Tulis ini di laporan.

In [ ]:
import os, platform, json
print("CPU logical :", os.cpu_count())
print("OS          :", platform.platform())
print("Python      :", platform.python_version())

In [ ]:
# === matmul_parallel: serial dan paralel (data parallelism) ===
import numpy as np
from multiprocessing import Pool, shared_memory


def kernel_baris(baris_a, bt):
    """Hitung sekelompok baris C. baris_a: list baris A, bt: B transpose (list)."""
    return [[sum(x * y for x, y in zip(ra, kb)) for kb in bt] for ra in baris_a]


def matmul_serial(A, B):
    return np.array(kernel_baris(A.tolist(), B.T.tolist()))


def _worker(args):
    nama_a, nama_b, nama_c, n, r0, r1 = args
    sa = shared_memory.SharedMemory(name=nama_a)
    sb = shared_memory.SharedMemory(name=nama_b)
    sc = shared_memory.SharedMemory(name=nama_c)
    A = np.ndarray((n, n), dtype=np.float64, buffer=sa.buf)
    B = np.ndarray((n, n), dtype=np.float64, buffer=sb.buf)
    C = np.ndarray((n, n), dtype=np.float64, buffer=sc.buf)
    C[r0:r1, :] = kernel_baris(A[r0:r1].tolist(), B.T.tolist())  # tiap proses menulis barisnya sendiri
    for s in (sa, sb, sc):
        s.close()


def matmul_parallel(A, B, p):
    n = A.shape[0]
    shm = [shared_memory.SharedMemory(create=True, size=A.nbytes) for _ in range(3)]
    try:
        for s, M in zip(shm[:2], (A, B)):
            np.ndarray(M.shape, dtype=np.float64, buffer=s.buf)[:] = M
        batas = np.linspace(0, n, p + 1, dtype=int)  # pembagian baris
        tugas = [(shm[0].name, shm[1].name, shm[2].name, n, int(batas[i]), int(batas[i + 1]))
                 for i in range(p)]
        with Pool(p) as pool:
            pool.map(_worker, tugas)
        return np.ndarray(A.shape, dtype=np.float64, buffer=shm[2].buf).copy()
    finally:
        for s in shm:
            s.close()
            s.unlink()

In [ ]:
# Uji cepat: hasil harus benar (True)
rng = np.random.default_rng(42)
A, B = rng.random((100, 100)), rng.random((100, 100))
print("Serial benar :", np.allclose(matmul_serial(A, B), A @ B))
print("Paralel benar:", np.allclose(matmul_parallel(A, B, 2), A @ B))

In [ ]:
import csv, statistics, time
import matplotlib.pyplot as plt
def ukur(fungsi, ulang=3):
    fungsi()  # pemanasan
    waktu = []
    for _ in range(ulang):
        t = time.perf_counter(); fungsi(); waktu.append(time.perf_counter() - t)
    return statistics.median(waktu)


def data(n):
    rng = np.random.default_rng(42)
    return rng.random((n, n)), rng.random((n, n))

N_A = 300
P_B = 8
ukuran_B = [100, 200, N_A]
baris = []
print(f"CPU logical: {os.cpu_count()}")

# Eksperimen A
A, B = data(N_A); acuan = A @ B
ts = ukur(lambda: matmul_serial(A, B))
baris.append(["A", N_A, 1, ts, 1.0, 100.0, 0.0, True])
print(f"[A] serial {ts:.4f}s")
for p in (2, 4, 8):
    tp = ukur(lambda: matmul_parallel(A, B, p))
    ok = bool(np.allclose(matmul_parallel(A, B, p), acuan))
    sp = ts / tp
    baris.append(["A", N_A, p, tp, sp, sp / p * 100, (ts - tp) / ts * 100, ok])
    print(f"[A] p={p} {tp:.4f}s speedup {sp:.3f}x efisiensi {sp/p*100:.1f}% benar={ok}")

# Eksperimen B
for n in ukuran_B:
    A, B = data(n); acuan = A @ B
    ts = ukur(lambda: matmul_serial(A, B))
    tp = ukur(lambda: matmul_parallel(A, B, P_B))
    ok = bool(np.allclose(matmul_parallel(A, B, P_B), acuan))
    sp = ts / tp
    baris.append(["B", n, P_B, tp, sp, sp / P_B * 100, (ts - tp) / ts * 100, ok, ts])
    print(f"[B] N={n} serial {ts:.4f}s paralel {tp:.4f}s speedup {sp:.3f}x benar={ok}")

with open("hasil_benchmark.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["eksperimen", "N", "proses", "waktu_detik", "speedup", "efisiensi_persen", "pengurangan_waktu_persen", "hasil_sama", "waktu_serial_B"])
    w.writerows(baris)
with open("info_sistem.json", "w") as f:
    json.dump({"os": platform.platform(), "python": platform.python_version(), "cpu": platform.processor(),
               "cpu_logical": os.cpu_count(), "numpy": np.__version__}, f, indent=2)

# Grafik
A_ = [b for b in baris if b[0] == "A"]; B_ = [b for b in baris if b[0] == "B"]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar([str(b[2]) for b in A_], [b[3] for b in A_], color="#1d4ed8")
ax[0].set(title="Waktu eksekusi", xlabel="Jumlah proses (1 = serial)", ylabel="Detik")
ax[1].plot([b[2] for b in A_], [b[4] for b in A_], "o-", label="Hasil pengujian")
ax[1].plot([1, 8], [1, 8], "--", color="gray", label="Ideal")
ax[1].set(title="Speedup", xlabel="Jumlah proses", ylabel="Speedup"); ax[1].legend()
ax[2].plot([b[1] for b in B_], [b[4] for b in B_], "o-", color="#0fa3b1")
ax[2].axhline(1, ls=":", color="gray")
ax[2].set(title=f"Speedup vs ukuran matriks (p={P_B})", xlabel="Ukuran N", ylabel="Speedup")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig("grafik_hasil.png", dpi=150); plt.show()
print("Selesai: hasil_benchmark.csv, info_sistem.json, grafik_hasil.png")

In [ ]:
from google.colab import files
for nama in ("hasil_benchmark.csv", "info_sistem.json", "grafik_hasil.png"):
    files.download(nama)